In [33]:
# ==============================================================================
# CÉLULA 1B: SETUP E CONFIGURAÇÕES INICIAIS
# ==============================================================================
import sys
import os
import logging

# 1. Configuração do Diretório (Garante conexão com o Drive se rodar no Colab)
try:
    from google.colab import drive
    drive.mount('/content/drive', force_remount=True)
    caminho_projeto = '/content/drive/MyDrive/1) PESQUISA/ESALQ Data Science/tcc/tema_classificacao_queda_arvore/git/tcc_risco_queda_v-pub'
    sys.path.append(caminho_projeto)
    os.chdir(caminho_projeto)
    print("✅ Diretório Colab configurado:", os.getcwd())
except ImportError:
    # Se estiver rodando localmente (não no Colab)
    print("✅ Rodando em ambiente local:", os.getcwd())

Mounted at /content/drive
✅ Diretório Colab configurado: /content/drive/MyDrive/1) PESQUISA/ESALQ Data Science/tcc/tema_classificacao_queda_arvore/git/tcc_risco_queda_v-pub


In [38]:
import pandas as pd
import geopandas as gpd
!pip install pyogrio -q
!pip install keplergl -q

from src.spatial_utils import classificar_risco_operacional

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.4/18.4 MB 31.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.4/12.4 MB 92.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 65.1 MB/s eta 0:00:00


In [35]:
from datetime import datetime, timedelta

# Defina o nome do cenário atual aqui (sem espaços, use underlines)
nome_cenario = "geral"

# Garante a existência da data base e adiciona o nome do cenário ao final
if 'data_hora_base' not in globals():
    data_hora_base = datetime.now().strftime("_%Y%m%d-%H%M")

sufixo_data = f"{data_hora_base}_{nome_cenario}"

print(sufixo_data)

_20260331-1103_geral


In [36]:
# ==============================================================================
# CÉLULA 0: CARREGAMENTO DE DADOS E MODELO PARA INFERÊNCIA
# ==============================================================================
import geopandas as gpd
import pandas as pd
import joblib
import os

# Certifique-se de que o nome_cenario é exatamente o mesmo usado no notebook 02
nome_cenario = "cenario_completo_triade_v1"

print("📂 Carregando base espacial via Pyogrio (Alta velocidade)...")

# 1. Carrega a base completa (GeoPackage)
file_path = "data/processed/dataset_all_features_imputed_DEDUP.gpkg"
df_full = gpd.read_file(file_path, engine='pyogrio')
print(f"✅ Dados carregados: {len(df_full)} ruas processadas.")

# 2. Resgate do Modelo e Threshold
caminho_modelo = f'results/models/modelo_xgb_{nome_cenario}.pkl'
caminho_threshold = f'results/models/threshold_{nome_cenario}.pkl'

if 'pipe_xgb_final' not in globals() or 'threshold_otimo' not in globals():
    print("🔄 Variáveis não estão na memória. Resgatando do disco...")

    if os.path.exists(caminho_modelo) and os.path.exists(caminho_threshold):
        pipe_xgb_final = joblib.load(caminho_modelo)
        threshold_otimo = joblib.load(caminho_threshold)
        print(f"✅ Modelo e Threshold ({threshold_otimo:.2f}) carregados do arquivo '.pkl' com sucesso!")
    else:
        print(f"❌ ERRO: Arquivos não encontrados em 'results/models/'.")
        print("Volte no Notebook 02 e rode a célula de exportação (joblib.dump).")
else:
    print(f"✅ Modelo e Threshold ({threshold_otimo:.2f}) já estão na memória ativos.")

📂 Carregando base espacial via Pyogrio (Alta velocidade)...
✅ Dados carregados: 110957 ruas processadas.
✅ Modelo e Threshold (0.46) já estão na memória ativos.


In [37]:
# ==============================================================================
# CÉLULA 1: INFERÊNCIA ESPACIAL (APLICAÇÃO DO MODELO NA CIDADE INTEIRA)
# ==============================================================================
print("🌍 Iniciando a varredura preditiva na malha viária completa de São Paulo...")

# 1. Preparação dos Dados (Usamos df_full original que já tem a geometria)
colunas_ignorar = [
    'cvc_nomelg', 'cvc_tplogr', 'cvc_classe',
    'target_historico_quedas', 'geometry', 'target_queda_bool'
]

# Separamos apenas as features para o modelo ler
X_cidade_inteira = df_full.drop(columns=colunas_ignorar, errors='ignore')

# 2. Geração de Probabilidades (XGBoost)
print("⚙️ Calculando probabilidades com o XGBoost Otimizado...")
df_full['prob_queda_xgb'] = pipe_xgb_final.predict_proba(X_cidade_inteira)[:, 1]

# 3. Classificação em Faixas de Risco (Usando o src/)
print(f"🚥 Classificando níveis operacionais baseados no limiar ótimo ({threshold_otimo:.2f})...")
df_full['classe_risco'] = classificar_risco_operacional(df_full['prob_queda_xgb'], threshold_otimo)

print("✅ Predições concluídas e acopladas à base espacial!")

🌍 Iniciando a varredura preditiva na malha viária completa de São Paulo...
⚙️ Calculando probabilidades com o XGBoost Otimizado...
🚥 Classificando níveis operacionais baseados no limiar ótimo (0.46)...
✅ Predições concluídas e acopladas à base espacial!


In [39]:
# ==============================================================================
# CÉLULA 2: EXPORTAÇÃO WEB DE ALTA PERFORMANCE (KEPLER.GL / UBER)
# ==============================================================================
from keplergl import KeplerGl
import os

print("🚀 Gerando mapa interativo WebGL de alta performance (Kepler.gl)...")

# 1. Prepara a base com todas as vias
colunas_web = ['cvc_tplogr', 'cvc_nomelg', 'prob_queda_xgb', 'classe_risco', 'geometry']
gdf_webmap = df_full[colunas_web].copy()

# O Kepler.gl exige EPSG:4326 (Lat/Lon)
gdf_webmap = gdf_webmap.to_crs(epsg=4326)

# 2. Inicializa o Mapa do Kepler.gl
# Por padrão ele abre em um estilo "Dark Mode" lindo para visualização de dados
mapa_kepler = KeplerGl(height=800)

# 3. Adiciona as mais de 100k vias ao mapa
mapa_kepler.add_data(data=gdf_webmap, name='Risco de Queda de Árvores')

# 4. Configuração visual do Kepler (pode ser ajustada direto na interface web depois)
# O Kepler vai colorir e permitir o filtro automaticamente na interface web dele.

# 5. Exporta para HTML
os.makedirs('results/maps', exist_ok=True)
caminho_html = f'results/maps/mapa_risco_kepler_SP.html'

mapa_kepler.save_to_html(file_name=caminho_html)

print(f"✅ Mapa WebGL gerado com sucesso: {caminho_html}")

🚀 Gerando mapa interativo WebGL de alta performance (Kepler.gl)...
User Guide: https://docs.kepler.gl/docs/keplergl-jupyter
Map saved to results/maps/mapa_risco_kepler_SP.html!
✅ Mapa WebGL gerado com sucesso: results/maps/mapa_risco_kepler_SP.html


In [ ]:
# ==============================================================================
# CÉLULA 2: EXPORTAÇÃO WEB KEPLER.GL (VERSÃO OTIMIZADA/LEVE)
# ==============================================================================
from keplergl import KeplerGl
import os

print("🚀 Gerando mapa interativo WebGL otimizado...")

colunas_web = ['cvc_tplogr', 'cvc_nomelg', 'prob_queda_xgb', 'classe_risco', 'geometry']
gdf_webmap = df_full[colunas_web].copy()

# 1. O FILTRO DE FOCO (A Mágica do Emagrecimento)
print("🧹 Removendo vias de 'Baixo Risco' para focar apenas nas áreas de atenção...")
gdf_webmap_leve = gdf_webmap[gdf_webmap['classe_risco'] != '1 - Baixo Risco'].copy()

# 2. Simplificação e Reprojeção
print("✂️ Simplificando geometrias...")
gdf_webmap_leve['geometry'] = gdf_webmap_leve.geometry.simplify(2.0)
gdf_webmap_leve = gdf_webmap_leve.to_crs(epsg=4326)

# 3. Geração do HTML
mapa_kepler = KeplerGl(height=800)
mapa_kepler.add_data(data=gdf_webmap_leve, name='Risco de Queda (Moderado a Crítico)')

os.makedirs('results/maps', exist_ok=True)
caminho_html = f'results/maps/mapa_risco_kepler_SP.html'

# Sobrescreve o arquivo gigante pelo novo arquivo leve
mapa_kepler.save_to_html(file_name=caminho_html)

tamanho_mb = os.path.getsize(caminho_html) / (1024 * 1024)
print(f"✅ Mapa gerado com sucesso! Tamanho final: {tamanho_mb:.2f} MB")
if tamanho_mb > 90:
    print("⚠️ Atenção: O arquivo ainda está perigosamente perto do limite do GitHub.")

🚀 Gerando mapa interativo WebGL otimizado...
🧹 Removendo vias de 'Baixo Risco' para focar apenas nas áreas de atenção...
✂️ Simplificando geometrias...
User Guide: https://docs.kepler.gl/docs/keplergl-jupyter


In [ ]:
# ==============================================================================
# CÉLULA 2: OTIMIZAÇÃO EXTREMA E EXPORTAÇÃO WEB (QGIS2WEB)
# ==============================================================================
import os

print("📦 Preparando arquivo otimizado para Webmapping...")

colunas_web = [
    'cvc_tplogr', 'cvc_nomelg',
    'prob_queda_xgb',
    'classe_risco',
    'geometry'
]

gdf_webmap = df_full[colunas_web].copy()

# 1. Formatação para o Popup
gdf_webmap['endereco_busca'] = gdf_webmap['cvc_tplogr'].astype(str) + " " + gdf_webmap['cvc_nomelg'].astype(str)
gdf_webmap['prob_formatada'] = (gdf_webmap['prob_queda_xgb'] * 100).round(1).astype(str) + "%"

gdf_webmap = gdf_webmap[['endereco_busca', 'prob_formatada', 'classe_risco', 'geometry']]

# ==========================================================
# A MÁGICA DO DESEMPENHO ACONTECE AQUI
# ==========================================================
print("✂️ Simplificando geometrias para reduzir o peso do arquivo...")
# Simplifica com 1 metro de tolerância ANTES de mudar a projeção (pois o EPSG original é em metros)
gdf_webmap['geometry'] = gdf_webmap.geometry.simplify(1.0)

print("🌐 Reprojetando coordenadas para WGS84 (Padrão Web)...")
# O EPSG:4326 é pesado de gerar em bases grandes, mas a simplificação anterior deixa isso muito mais rápido
gdf_webmap = gdf_webmap.to_crs(epsg=4326)

# 2. Exportação
os.makedirs('results/maps', exist_ok=True)
caminho_exportacao = f'results/maps/mapa_risco_qgis2web_{sufixo_data}.geojson'

# Usamos o engine 'pyogrio' também para salvar, é infinitamente mais rápido que o padrão
gdf_webmap.to_file(caminho_exportacao, driver='GeoJSON', engine='pyogrio')

print(f"✅ Arquivo LEVE exportado com sucesso para: {caminho_exportacao}")

📦 Preparando arquivo otimizado para Webmapping...
✂️ Simplificando geometrias para reduzir o peso do arquivo...
🌐 Reprojetando coordenadas para WGS84 (Padrão Web)...
✅ Arquivo LEVE exportado com sucesso para: results/maps/mapa_risco_qgis2web__20260331-1103_geral.geojson
